# 1  Preparing visa application records

**Goal:** turn raw records into data a model can use. You will find the problems, split the data first, and decide for each problem what to do.

**Data:** 5,000 synthetic visa applications, entered by hand at different counters and online.

**The data used here:**
- `visa_applications_raw.csv`: [view as a table](https://github.com/u1i/ml-foundations-workshop/blob/main/data/visa_applications_raw.csv) (opens on GitHub) · [download](https://raw.githubusercontent.com/u1i/ml-foundations-workshop/main/data/visa_applications_raw.csv) (right-click, *Save link as*)

**How this works:** each step is one cell. Click the &#9654; button on the left of the cell and read what appears below it. Where a cell has a field on the right, choose an option and click &#9654; again. No coding is needed. The cells marked *Stretch* at the end are optional and for those who like to code.

All data is synthetic.

In [ ]:
#@title Setup  (Click &#9654; first)
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display, HTML
DATA = "https://raw.githubusercontent.com/u1i/ml-foundations-workshop/main/data/"
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"font.size": 13, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
RED, INK, GREY, GREEN, BLUE = "#A12527", "#1d1d1b", "#6e6e6a", "#2e7d4f", "#1f5a8a"
_STYLE = {"info": ("#1f5a8a", "#eef4f9"), "see": ("#A12527", "#fbeeee"), "ok": ("#2e7d4f", "#e9f5ee"),
          "try": ("#9a6a00", "#fbf3dd"), "step": ("#1d1d1b", "#f6f5f2")}

def box(title, text="", kind="info"):
    c, bg = _STYLE[kind]
    display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;border-left:6px solid {c};background:{bg};'
                 f'color:#1d1d1b;padding:14px 20px;margin:12px 0;font-size:17px;line-height:1.5;max-width:900px">'
                 f'<div style="font-size:20px;font-weight:bold;color:{c};margin-bottom:4px">{title}</div>{text}</div>'))

def big(*items):
    """big((value, label), (value, label, colour), ...): large numbers in one row."""
    cells = "".join(f'<div style="display:inline-block;margin:6px 40px 6px 0;vertical-align:top"><div style="font-size:15px;color:#6e6e6a">{it[1]}</div>'
                    f'<div style="font-size:44px;font-weight:bold;color:{it[2] if len(it) > 2 else INK}">{it[0]}</div></div>' for it in items)
    display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;color:#1d1d1b">{cells}</div>'))

def answer(choice, options):
    """options: {option: (is_right, explanation)}; shows feedback for the chosen option."""
    if choice not in options:
        box("Make your choice", "Pick an option in the field on the right of this cell, then click &#9654; again.", "try")
        return False
    right, why = options[choice]
    box("Correct" if right else "Not quite", why, "ok" if right else "try")
    return right

def table(df, highlight=None, n=10):
    """Show rows with cells highlighted: highlight = {(row_label, column): colour}."""
    sub = df.head(n).astype(object).where(df.head(n).notna(), "(empty)")
    def colour(_):
        out = pd.DataFrame("", index=sub.index, columns=sub.columns)
        for (r, c), col in (highlight or {}).items():
            if r in out.index and c in out.columns:
                out.loc[r, c] = f"background-color:{col};font-weight:bold"
        return out
    display(sub.style.apply(colour, axis=None).set_table_styles([{"selector": "td,th", "props": "font-size:14px;padding:4px 10px"}]))
import ipywidgets as W
def fslider(desc, lo, hi, step, val, fmt=".2f"):
    return W.FloatSlider(value=val, min=lo, max=hi, step=step, description=desc, continuous_update=False, readout_format=fmt,
                         layout=W.Layout(width="700px"), style={"description_width": "200px"})
def islider(desc, lo, hi, val):
    return W.IntSlider(value=val, min=lo, max=hi, description=desc, continuous_update=False,
                       layout=W.Layout(width="700px"), style={"description_width": "200px"})
live = W.interact

def matrix(y_true, y_flag, labels=("honest", "fraud"), flag_word="flagged"):
    """Confusion matrix as four large boxes."""
    y_true, y_flag = np.asarray(y_true).astype(bool), np.asarray(y_flag).astype(bool)
    tp, fn = int((y_true & y_flag).sum()), int((y_true & ~y_flag).sum())
    fp, tn = int((~y_true & y_flag).sum()), int((~y_true & ~y_flag).sum())
    cell = lambda n, t, c, bg: (f'<td style="border:2px solid #1d1d1b;padding:14px 18px;background:{bg};width:260px">'
                                f'<div style="font-size:15px;color:{c};font-weight:bold">{t}</div><div style="font-size:38px;font-weight:bold;color:{c}">{n:,}</div></td>')
    display(HTML(f'<table style="font-family:Arial,Helvetica,sans-serif;border-collapse:collapse;color:#1d1d1b;margin:8px 0">'
                 f'<tr><td></td><th style="padding:6px;font-size:15px">Model: {flag_word}</th><th style="padding:6px;font-size:15px">Model: not {flag_word}</th></tr>'
                 f'<tr><th style="padding:6px;font-size:15px;text-align:right">Really {labels[1]}</th>{cell(tp, "Found", GREEN, "#e9f5ee")}{cell(fn, "Missed", RED, "#fbeeee")}</tr>'
                 f'<tr><th style="padding:6px;font-size:15px;text-align:right">Really {labels[0]}</th>{cell(fp, "Checked for nothing", RED, "#fbeeee")}{cell(tn, "Correctly left alone", GREEN, "#e9f5ee")}</tr></table>'))
    return tp, fn, fp, tn

def bars(values, title, highlight=None, fmt="{:.2f}", color=INK):
    """Horizontal bars with values; values is a dict or Series."""
    s = pd.Series(values)
    fig, ax = plt.subplots(figsize=(9, 0.5 * len(s) + 1))
    ax.barh(s.index.astype(str), s.values, color=[RED if k == highlight else color for k in s.index])
    for i, v in enumerate(s.values): ax.text(v, i, " " + fmt.format(v), va="center", fontsize=12)
    ax.invert_yaxis(); ax.set_title(title); plt.show()


box("Ready", "Setup done. Continue with Step 1 below.", "ok")

## Step 1  Look at the raw data

In [ ]:
#@title Step 1  Load the raw records
raw = pd.read_csv(DATA + "visa_applications_raw.csv")
df = raw.drop_duplicates().copy()
sample = df.iloc[:45]
hl = {}
for i, row in sample.iterrows():
    if pd.isna(row["applicant_age"]) or "years" in str(row["applicant_age"]) or str(row["applicant_age"]) == "999": hl[(i, "applicant_age")] = "#fbd9d9"
    if pd.isna(row["channel"]): hl[(i, "channel")] = "#fbd9d9"
    if str(row["visa_type"]) not in ["Tourist", "Employment", "Family", "Student"]: hl[(i, "visa_type")] = "#fbd9d9"
    if str(row["documents_complete"]) not in ["Yes", "No"]: hl[(i, "documents_complete")] = "#fbd9d9"
show = sample[[r in {k[0] for k in hl} for r in sample.index]].head(12)
big((f"{len(raw):,}", "rows loaded"), (raw.duplicated().sum(), "exact duplicates, removed", RED))
table(show[["application_id", "submission_date", "visa_type", "channel", "applicant_age", "documents_complete", "processing_days"]], hl, n=12)
box("What you see", "Some of the records. The red cells are problems a model cannot handle: empty cells, an age written as text, "
    "the same visa type written in different ways, and Yes/No written as Y, 1 or yes.", "see")

In [ ]:
#@title Step 1b  How often does each problem occur?
missing = df.isna().sum()
spellings = df["visa_type"].nunique()
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
missing[missing > 0].sort_values().plot.barh(ax=ax[0], color=RED); ax[0].set_title("Empty cells per column")
df["visa_type"].value_counts().plot.barh(ax=ax[1], color=[RED if v not in ["Tourist","Employment","Family","Student"] else INK for v in df["visa_type"].value_counts().index])
ax[1].set_title(f"visa_type: {spellings} different spellings for 4 types"); ax[1].invert_yaxis()
plt.tight_layout(); plt.show()
box("Do you see it?", f"The column <b>visa_type</b> has {spellings} different values for only four visa types, in English and Arabic. "
    "To a model, <i>Tourist</i>, <i>TOURIST</i> and <i>سياحة</i> are three different things.", "see")

## Step 2  Your first decision

In [ ]:
#@title Step 2  Quick check: which problem is this?  (choose an option on the right, then click &#9654;)
question = "Choose..." #@param ["Choose...", "Missing value", "Wrong type", "Outlier", "Inconsistent category"]
box("The cell shows: applicant_age = '32 years'", "Which kind of problem is it?", "step")
_ = answer(question, {
    "Missing value": (False, "The cell is not empty. It holds a value, just in the wrong form."),
    "Wrong type": (True, "Age should be a number. '32 years' is text, so the model cannot calculate with it. We will strip the word and keep 32."),
    "Outlier": (False, "32 is a normal age. The problem is the word 'years', which turns the number into text."),
    "Inconsistent category": (False, "Age is a number, not a category. The problem is that it is stored as text."),
})

## Step 3  Split the data first

In [ ]:
#@title Step 3  Split into training and test data
from sklearn.model_selection import train_test_split
train, test = train_test_split(df, test_size=0.2, random_state=42)
fig, ax = plt.subplots(figsize=(11, 1.4))
ax.barh([0], [len(train)], color=INK); ax.barh([0], [len(test)], left=[len(train)], color="#bbbbbb")
ax.text(len(train) / 2, 0, f"Training: {len(train):,}", color="white", ha="center", va="center", fontsize=15, weight="bold")
ax.text(len(train) + len(test) / 2, 0, f"Test: {len(test):,}", ha="center", va="center", fontsize=15, weight="bold")
ax.axis("off"); plt.show()
box("Why first?", "Some preparation steps learn from the data, for example the value used to fill an empty age. "
    "If they learn from the test data too, the test result looks better than the model really is. "
    "So from now on, every step learns from the <b>training data only</b> and is then applied to both.", "info")

## Step 4  Fix types and spelling

In [ ]:
#@title Step 4  Apply the spelling and type rules
TYPE_MAP = {"tourist": "Tourist", "سياحة": "Tourist", "work": "Employment", "employment": "Employment", "عمل": "Employment", "شغل": "Employment",
            "family": "Family", "family visit": "Family", "عائلية": "Family", "student": "Student", "دراسة": "Student", "طالب": "Student"}
DOCS_MAP = {"yes": "Yes", "y": "Yes", "1": "Yes", "no": "No", "n": "No", "0": "No"}
def fix(d):
    d = d.copy()
    d["visa_type"] = d["visa_type"].str.strip().str.lower().map(TYPE_MAP)
    d["documents_complete"] = d["documents_complete"].astype(str).str.strip().str.lower().map(DOCS_MAP)
    d["applicant_age"] = pd.to_numeric(d["applicant_age"].astype(str).str.replace(" years", ""), errors="coerce")
    d["submission_date"] = pd.to_datetime(d["submission_date"], format="mixed", dayfirst=True)
    return d
before = train["visa_type"].nunique()
train, test = fix(train), fix(test)
fig, ax = plt.subplots(figsize=(7, 3))
train["visa_type"].value_counts().plot.barh(ax=ax, color=INK); ax.invert_yaxis(); ax.set_title("visa_type after the rules")
plt.show()
big((before, "spellings before", RED), (train["visa_type"].nunique(), "visa types after", GREEN))
box("Do you see it now?", f"{before} spellings became four visa types. Arabic and English entries now count as the same type, "
    "so the model can learn from all of them together. Dates written as 13/11/2025 and 2025-11-13 are also read the same way now.", "see")

## Step 5  Outliers: error or real case?

In [ ]:
#@title Step 5a  An applicant aged 999  (choose, then click &#9654;)
decision = "Choose..." #@param ["Choose...", "Error: treat it as missing", "Real case: keep it"]
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.scatter(train["applicant_age"], np.random.default_rng(1).random(len(train)), s=8, color=INK, alpha=.4)
ax.scatter(train.loc[train["applicant_age"] > 100, "applicant_age"], np.full((train["applicant_age"] > 100).sum(), .5), s=80, color=RED)
ax.set_yticks([]); ax.set_xlabel("applicant_age"); ax.set_title("Ages in the training data"); plt.show()
right = answer(decision, {
    "Error: treat it as missing": (True, "Nobody is 999 years old. It is an entry error, probably a placeholder. We set it to empty and fill it in Step 6."),
    "Real case: keep it": (False, "An age of 999 cannot be real. Keeping it would distort everything the model learns about age."),
})
for d in (train, test):
    d.loc[d["applicant_age"] > 100, "applicant_age"] = np.nan

In [ ]:
#@title Step 5b  An application that took 157 days  (choose, then click &#9654;)
decision = "Choose..." #@param ["Choose...", "Error: remove it", "Real case: keep it"]
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.hist(train["processing_days"], bins=80, color=INK); ax.set_xlabel("processing days"); ax.set_title("Processing time in the training data")
ax.annotate("157 days", xy=(157, 2), xytext=(120, 300), color=RED, fontsize=14, arrowprops=dict(color=RED)); plt.show()
_ = answer(decision, {
    "Error: remove it": (False, "The file shows it was put on hold for an additional document review. Such cases happen. If we remove them, the model will be surprised by them in the service."),
    "Real case: keep it": (True, "Long cases are rare but real: files on hold for an additional document review. The model will meet them in the service, so they stay."),
})

## Step 6  Missing values: drop, fill or flag

In [ ]:
#@title Step 6a  Empty ages  (choose, then click &#9654;)
decision = "Choose..." #@param ["Choose...", "Drop the records", "Fill with the median age of the training data", "Add a flag column"]
n_missing = train["applicant_age"].isna().sum()
big((n_missing, "training records without age", RED), (f"{n_missing / len(train):.0%}", "of the training data"))
ok = answer(decision, {
    "Drop the records": (False, f"That would throw away {n_missing} applications, and the gap looks random: the age field was optional on one form. Filling keeps them."),
    "Fill with the median age of the training data": (True, "The gap is random and age matters, so we fill it with a typical value: the median, taken from the training data only."),
    "Add a flag column": (False, "A flag helps when the gap itself carries information. Here the field was simply optional, so a flag adds nothing."),
})
median_age = train["applicant_age"].median()
train["applicant_age"] = train["applicant_age"].fillna(median_age); test["applicant_age"] = test["applicant_age"].fillna(median_age)
if ok: box("Done", f"Empty ages filled with {median_age:.0f}, the median of the training data, in both parts.", "ok")

In [ ]:
#@title Step 6b  Empty channel  (choose, then click &#9654;)
decision = "Choose..." #@param ["Choose...", "Drop the records", "Fill with the most common channel", "Add a flag column and write 'Unknown'"]
big((train["channel"].isna().sum(), "training records without channel", RED))
ok = answer(decision, {
    "Drop the records": (False, "These applications would be lost, and the gap may tell us something: they often come from older systems."),
    "Fill with the most common channel": (False, "That would hide the fact that the channel was not recorded, which can matter for delays."),
    "Add a flag column and write 'Unknown'": (True, "The gap may carry information, so we keep it visible: a new column channel_missing, and 'Unknown' as the channel."),
})
for d in (train, test):
    d["channel_missing"] = d["channel"].isna().astype(int); d["channel"] = d["channel"].fillna("Unknown")
train = train.dropna(subset=["supporting_document_count"]); test = test.dropna(subset=["supporting_document_count"])
if ok: box("Done", "Flag column added. The few rows without a document count were dropped: a random gap, very few records.", "ok")

## Step 7  Encode categories

In [ ]:
#@title Step 7  Turn categories into numbers
cat_cols = ["visa_type", "applicant_region", "channel", "sponsor_type", "documents_complete", "additional_document_review_required"]
X_train = pd.get_dummies(train[cat_cols], dtype=int)
X_test = pd.get_dummies(test[cat_cols], dtype=int).reindex(columns=X_train.columns, fill_value=0)
display(pd.concat([train[["visa_type"]].head(5), X_train[[c for c in X_train.columns if c.startswith("visa_type")]].head(5)], axis=1))
box("What happened", "Each visa type became its own column of 0 and 1. A model can work with these numbers. "
    "The test data gets exactly the same columns as the training data.", "info")

## Step 8  Result

In [ ]:
#@title Step 8  Check the result
left = int(train[["applicant_age", "channel", "visa_type", "documents_complete"]].isna().sum().sum())
big((f"{len(train):,}", "training records ready", GREEN), (f"{len(test):,}", "test records, prepared the same way"), (left, "problems left", GREEN if left == 0 else RED))
box("Do you see it now?", f"The raw records had empty cells, text where numbers belong, {spellings} spellings for four visa types and an impossible age. "
    f"Now every record has a clean value in every column. <b>A model can now learn from all {len(train) + len(test):,} applications</b>, including the Arabic entries and the ones with gaps. "
    "Every decision you made came from what the value means, not from a rule of thumb.", "see")

---
## Stretch (optional, for those who code)

In [ ]:
# Stretch: find two more data quality problems in the raw data and fix them (look at each column)
for col in raw.columns:
    print(col, raw[col].nunique(), "different values;", raw[col].isna().sum(), "empty")
# for example: raw["supporting_document_count"].describe(), raw["previous_visits"].value_counts()